The same lab as a web page, with the folded answers: https://ml.wxing.me/materials/lesson10/lab.html

# Lab 10: Use a CNN, and Catch It When It Goes Wrong
## MPS311/439 Machine Learning
**Lesson 10 Lab Session | Core: about 45 minutes**

## What this lab is for

The lecture explained why a convolutional network (CNN) uses the spatial structure of an image. Today you use one as a tool, on five jobs: give images to a CNN in the shape it wants, compare it fairly with the fully connected network, check by hand what one convolution layer computes, find where it goes wrong, and check code that an AI wrote. All the code is given. You run it, change the value marked `← change`, and decide what the numbers tell you. After each Part, **write down** one number from your output and one sentence.

Each Part ends with a folded **Going further** box: open questions with no single answer, for when you finish early. The folded **Try with AI** boxes are also for when you finish early; the **Advanced Study** is for after class.

## How to work

1. This notebook already has all the code. Save a copy first (**File → Save a copy in Drive**). Run the cells from top to bottom: later cells use names made by earlier ones. Keras is already installed in Colab; you do not need a GPU.
2. Write your answers in a text cell (**+ Text**) under the code.
3. **Run, and you should see** tells you what your output should show, so you know you are on track. Training uses random numbers, and Colab does the arithmetic in a slightly different order from the computer that made this page: a difference in the third decimal place is fine.
4. Nothing is shown while a network trains; that is normal. Each training cell says roughly how long it takes on Colab.

## Part 0: Set up (4 minutes)

Run this cell without reading it closely. The last line downloads MNIST, the digits of Assignment 2 (about 11 MB). If it stops with a download error, use the Appendix.

In [ ]:
import numpy as np                                        # numbers and arrays
import matplotlib.pyplot as plt                           # plots
import time                                               # a clock, to time the training
import urllib.request                                     # to download a file (only for the Appendix)
from sklearn.metrics import confusion_matrix              # the 10 x 10 table of right and wrong answers
from tensorflow import keras                              # neural networks

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()   # ← change: which dataset; x_test is not used today

The data block: the same split as Assignment 2. The CNN wants each image as 28 × 28 × 1; the fully connected network wants a row of 784 numbers. We keep both.

In [ ]:
X = x_train.reshape(-1, 28, 28, 1) / 255.0                # 60,000 images of 28 x 28 x 1, scaled to 0 to 1
y = y_train                                               # the digit, 0 to 9
X_fit = X[:50000]                                         # the first 50,000 images: fit the models
y_fit = y[:50000]                                         # and their digits
X_val = X[50000:]                                         # the last 10,000 images: compare the models
y_val = y[50000:]                                         # and their digits
X_fit_flat = X_fit.reshape(-1, 784)                       # the same images as rows of 784 numbers
X_val_flat = X_val.reshape(-1, 784)                       # the same for the validation images
print(X_fit.shape, X_fit_flat.shape, X_val.shape)         # the shapes

**Run, and you should see:** `(50000, 28, 28, 1) (50000, 784) (10000, 28, 28, 1)`.

## Part 1: Give the images to a CNN (6 minutes)

The MPS439 extension of Assignment 2 asks for a CNN with "input 28 × 28 × 1, at least two `Conv2D` layers". Build the lecture's CNN, and see what happens if you give it the wrong shape.

In [ ]:
keras.utils.set_random_seed(0)                            # fix the random start
cnn = keras.Sequential([
    keras.layers.Input((28, 28, 1)),                      # height, width, one image
    keras.layers.Conv2D(16, 3, activation="relu"),        # ← change: the number of filters (16), each 3 x 3
    keras.layers.MaxPooling2D(2),                         # keep the largest of each 2 x 2
    keras.layers.Conv2D(32, 3, activation="relu"),        # 32 filters, each looking at all the feature maps
    keras.layers.MaxPooling2D(2),                         # keep the largest of each 2 x 2
    keras.layers.Flatten(),                               # all the numbers in one row
    keras.layers.Dense(10, activation="softmax"),         # one probability per digit
])
cnn.summary()                                             # the shape and the parameters of each layer
probs = cnn.predict(X_val_flat[:5], verbose=0)            # ← change: X_val_flat or X_val
print(probs.shape)                                        # 5 images, 10 probabilities each

The red error message is expected. Read its last lines: which shape did the CNN expect, and which did it get? Then change `X_val_flat` to `X_val` and run again.

**Run, and you should see:** first an error ending in `Expected shape (None, 28, 28, 1), but input has incompatible shape (5, 784)`; after the change, the table of layers with `Total params: 12,810` and then `(5, 10)`.

Now count the parameters (weights plus constants) by hand:

In [ ]:
first = 16 * (3 * 3 + 1)                                  # 16 filters, each 9 weights and one constant
second = 32 * (3 * 3 * 16 + 1)                            # 32 filters, each 3 x 3 on all 16 feature maps, and one constant
last = (5 * 5 * 32) * 10 + 10                             # 800 numbers to 10 units, plus 10 constants
print(first, second, last, first + second + last)         # the three layers and the total

**Run, and you should see:** `160 4640 8010 12810`, the same as the `Param #` column of `cnn.summary()`.

Change the 16 in the first `Conv2D` to 8, run the first cell again, and read the new total. **Then change it back to 16 and run the cell once more:** Part 2 uses this network.

**Write down:** the total number of parameters with 8 filters. In one sentence: which two layers changed, and why does the second one change too?

*Your answer:* 

> **Going further (optional)**
>
> - Change the size of the first filters from 3 to 5. Which output shapes change in `cnn.summary()`, and why does 28 become 24?
> - Add `padding="same"` to the first `Conv2D`. What happens to its output shape and to its number of parameters?


## Part 2: Compare the two networks fairly (11 minutes)

Table A of the assignment asks for two networks with the same data, the same number of epochs and the same seed: the number of parameters, the validation accuracy, the training time (seconds taken by `fit`), and the accuracy after moving the images 2 pixels to the right. This cell trains the fully connected network, then the CNN of Part 1, for 2 epochs each: about 20 seconds, then 1 to 2 minutes on Colab.

In [ ]:
keras.utils.set_random_seed(0)                            # fix the random start
dense = keras.Sequential([
    keras.layers.Input((784,)),                           # 784 pixels in a row
    keras.layers.Dense(128, activation="relu"),           # hidden layer 1
    keras.layers.Dense(128, activation="relu"),           # hidden layer 2
    keras.layers.Dense(10, activation="softmax"),         # one probability per digit
])
dense.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # the fully connected network
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])     # the CNN of Part 1, the same way
epochs = 2                                                # ← change: the number of epochs, the same for both

start = time.time()                                       # start the clock
dense.fit(X_fit_flat, y_fit, epochs=epochs, verbose=0)    # train the fully connected network on rows
dense_seconds = time.time() - start                       # seconds taken by fit
start = time.time()                                       # start the clock again
cnn.fit(X_fit, y_fit, epochs=epochs, verbose=0)           # train the CNN on 28 x 28 x 1 images
cnn_seconds = time.time() - start                         # seconds taken by fit
print(round(dense_seconds), round(cnn_seconds))           # seconds

Now move every validation image to the right and test both networks. Neither network is trained again.

In [ ]:
move = 2                                                  # ← change: how many pixels to the right, 0 to 4
X_shift = np.zeros_like(X_val)                            # same shape as X_val, all paper (0)
X_shift[:, :, move:] = X_val[:, :, :28 - move]            # every image moved to the right
dense_acc = dense.evaluate(X_val_flat, y_val, verbose=0)[1]                      # validation accuracy
dense_moved = dense.evaluate(X_shift.reshape(-1, 784), y_val, verbose=0)[1]      # after the move, as rows
cnn_acc = cnn.evaluate(X_val, y_val, verbose=0)[1]                               # validation accuracy
cnn_moved = cnn.evaluate(X_shift, y_val, verbose=0)[1]                           # after the move, as images
print("fully connected", dense.count_params(), round(dense_acc, 4), round(dense_moved, 4))   # parameters, accuracy, moved
print("CNN", cnn.count_params(), round(cnn_acc, 4), round(cnn_moved, 4))                     # the same for the CNN

**Run, and you should see:** two times in seconds (the CNN's is several times the other's), then `fully connected 118282 0.9716 0.8263` and `CNN 12810 0.9824 0.9517`.

Change `move` to 4 and run the second cell again (a few seconds). The other moves are in Going further.

**Write down:** your four columns of Table A (parameters, validation accuracy, seconds, accuracy after a 2-pixel move) for both networks. In one sentence: what does the CNN buy, and what does it cost?

*Your answer:* 

**Think about** (with your neighbour, no writing): the CNN is 1.1 points more accurate. A change of seed alone moves the fully connected network by up to about 0.3 points. Is 1.1 points a real difference? (One way to judge: train both again with `keras.utils.set_random_seed(1)` and see how far each number moves.)

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the accuracy of both networks against the size of the move, 0 to 4 pixels, as two lines on one plot.
> 2. **Tell the AI:** `dense` takes rows of 784 numbers, `cnn` takes images of 28 × 28 × 1, and the moved images are made as in the cell above (`X_shift[:, :, move:] = X_val[:, :, :28 - move]`).
> 3. **Ask:** "Write simple Python, one step per line, that loops over move = 0 to 4, makes the moved validation images, evaluates both networks and plots the two accuracies against the move."
> 4. **Check:** at a move of 2 the CNN is at 0.9517 and the fully connected network at 0.8263.

> **Going further (optional)**
>
> - Change `move` to 0, 1 and 3 as well. At which move does the CNN start to fail?
- Set `epochs = 5` and run both cells again (this takes a few minutes). Does the gap between the two networks grow or shrink? Assignment 2 uses 20 epochs for both.
> - The assignment asks which images the CNN gets right that the fully connected network gets wrong. Count them with `cnn.predict(X_val, verbose=0).argmax(axis=1)` and the same for `dense`. Are they the same images the other way round?


## Part 3: What does one convolution layer compute? (5 minutes)

The lecture said a convolution slides a 3 × 3 filter over the image, multiplies its 9 weights by the 9 pixels under it, adds them up with one constant, then sets negative numbers to 0 (ReLU). Check that this is exactly what the first layer of your trained CNN does: take one of its 16 learned filters, compute its feature map by hand, and compare with the one Keras computes.

In [ ]:
weights, constants = cnn.layers[0].get_weights()          # the first layer: 3 x 3 x 1 x 16 weights and 16 constants
k = 0                                                     # ← change: which filter, 0 to 15
f = weights[:, :, 0, k]                                   # its 3 x 3 weights
image = X_val[0, :, :, 0]                                 # the first validation image, 28 x 28
by_hand = np.zeros((26, 26))                              # one number per position of the filter
for i in range(26):                                       # every row of the feature map
    for j in range(26):                                   # every column of the feature map
        patch = image[i:i + 3, j:j + 3]                   # the 3 x 3 pixels under the filter
        by_hand[i, j] = np.sum(f * patch) + constants[k]  # multiply one by one, add up, add the constant
by_hand = np.maximum(by_hand, 0)                          # ReLU: negative numbers become 0
first_layer = keras.Model(cnn.inputs, cnn.layers[0].output)       # the CNN cut after its first layer
by_keras = first_layer.predict(X_val[:1], verbose=0)[0, :, :, k]  # the feature map Keras computes
print(np.abs(by_hand - by_keras).max())                   # the largest difference between the two

fig, axes = plt.subplots(1, 4, figsize=(12, 3))           # the image, the filter and the two feature maps
axes[0].imshow(image, cmap="Greys")
axes[0].set_title("image")
axes[1].imshow(f, cmap="RdBu_r", vmin=-np.abs(f).max(), vmax=np.abs(f).max())   # red positive, blue negative
axes[1].set_title("filter " + str(k))
axes[2].imshow(by_hand, cmap="Greys")
axes[2].set_title("by hand")
axes[3].imshow(by_keras, cmap="Greys")
axes[3].set_title("Keras")
plt.show()

**Run, and you should see:** a very small number such as `7.4e-08` (anything below 0.000001 is the same), then a 3, a 3 × 3 square of red and blue, and two feature maps that look the same.

Change `k` to two or three other filters and run again.

**Write down:** the largest difference for one filter. In one sentence: what does this filter seem to look for in the 3?

*Your answer:* 

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** all 16 learned filters of the first layer as small images.
> 2. **Tell the AI:** `weights` has shape 3 × 3 × 1 × 16: one 3 × 3 filter per position of the last axis.
> 3. **Ask:** "Show each of the 16 filters in `weights` as a 3 × 3 image in a 2 × 8 grid, with the colour map 'RdBu_r' centred at 0 and the title 'filter k'."
> 4. **Check:** 16 small red and blue squares in 2 rows of 8; filter 0 looks the same as in the cell above.

> **Going further (optional)**
>
> - The second layer has 32 filters of 3 × 3 × 16. How would you change the loop to compute one of its feature maps by hand? What would `patch` be now?


## Part 4: Where does the CNN go wrong? (6 minutes)

Section 4 of Assignment 2 asks for the 5 most common mistakes (Table 5) and 10 misclassified images (Figure 3). Here on the validation set, with the CNN of Part 2; the assignment does it once on `test` with your final model. The lecture showed a network trained for 5 epochs, so its most common mistakes differ from today's.

In [ ]:
pred = cnn.predict(X_val, verbose=0).argmax(axis=1)       # the most probable digit of each image
cm = confusion_matrix(y_val, pred)                        # rows: true digit, columns: predicted digit
print(cm)                                                 # the table
mistakes = cm.copy()                                      # a copy to keep only the mistakes
np.fill_diagonal(mistakes, 0)                             # the diagonal is right: set it to 0
for k in range(5):                                        # the 5 most common mistakes
    flat = mistakes.argmax()                              # position of the largest number, counted row by row
    i = flat // 10                                        # its row: the true digit
    j = flat % 10                                         # its column: the predicted digit
    print("true", i, "predicted", j, "images", mistakes[i, j])   # one line of Table 5
    mistakes[i, j] = 0                                    # remove it and look for the next one

wrong = np.where(pred != y_val)[0]                        # positions of the images it gets wrong
print(len(wrong))                                         # how many
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))         # 2 rows of 5 images
for k in range(10):                                       # the first 10 mistakes
    ax = axes[k // 5, k % 5]                              # row and column of this image
    ax.imshow(X_val[wrong[k], :, :, 0], cmap="Greys")     # the image
    ax.set_title("true " + str(y_val[wrong[k]]) + ", predicted " + str(pred[wrong[k]]))   # its two labels
    ax.axis("off")                                        # no axes
plt.show()

**Run, and you should see:** a 10 × 10 table; five lines of mistakes, the first `true 5 predicted 8 images 12` (tied with 7 read as 9); then `176` and 10 images with their labels.

**Write down:** the most common mistake and how many images it has. In one sentence: for how many of the 10 images could **you** read the true digit, and what does that say about where the mistakes come from?

*Your answer:* 

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the confusion matrix as a picture, with the mistakes only.
> 2. **Tell the AI:** `cm` is a 10 × 10 numpy array, rows the true digit and columns the predicted digit.
> 3. **Ask:** "Copy `cm`, set its diagonal to 0, and show it with matplotlib `imshow`, with the count written in every cell that is not 0, and axis labels 'predicted' and 'true'."
> 4. **Check:** the darkest cell is the first line printed by the loop above.

> **Going further (optional)**
>
> - Change `wrong[k]` to `wrong[k + 10]` in the three places it appears, to see the next 10 mistakes. Do the same pairs come back?
> - Make the same table for the fully connected network (`dense.predict(X_val_flat, ...)`). Are its most common mistakes the same pairs?


## Part 5: Check the code an AI wrote (9 minutes)

You asked an AI to "compare the CNN and the fully connected network, as in Table A". It gave three pieces of code. Each changes one line of what you ran in Parts 1 and 2. For each, decide: is it **wrong**, or **right**? Do not judge by the score alone. Put each version next to the Part 2 code and compare it **line by line**: which line is different, and is it still the same test? Run them in order, without running the Part 1 cell again (Version C needs the CNN trained in Part 2).

**Version A** (the move test)

In [ ]:
X_shift = np.zeros_like(X_val)                            # same shape as X_val, all paper (0)
X_shift[:, :, 2:] = X_val[:, :, :28 - 2]                  # every image moved 2 pixels to the right
dense_moved = dense.evaluate(X_val_flat, y_val, verbose=0)[1]      # the fully connected network, moved images
cnn_moved = cnn.evaluate(X_shift, y_val, verbose=0)[1]            # the CNN, moved images
print(round(dense_moved, 4), round(cnn_moved, 4))         # "the fully connected network does not mind a move"

**Version B** (the CNN)

In [ ]:
keras.utils.set_random_seed(0)                            # fix the random start
cnn_b = keras.Sequential([
    keras.layers.Input((28, 28, 1)),                      # height, width, one image
    keras.layers.Conv2D(16, 3, activation="relu"),        # 16 filters of 3 x 3
    keras.layers.MaxPooling2D(2),                         # keep the largest of each 2 x 2
    keras.layers.Conv2D(32, 3, activation="relu"),        # 32 filters of 3 x 3
    keras.layers.MaxPooling2D(2),                         # keep the largest of each 2 x 2
    keras.layers.Dense(10, activation="softmax"),         # one probability per digit
])
cnn_b.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in Part 2
cnn_b.fit(X_fit[:1000], y_fit[:1000], epochs=1, verbose=0)         # a quick try on 1,000 images

**Version C** (the training). Read it first. Running it trains for 1 to 2 minutes on Colab: run it only if you have the time, or after class.

In [ ]:
cnn.fit(X_fit, y_fit, epochs=2, verbose=0)                # "train the CNN for 2 epochs, as the fully connected network"
cnn_acc = cnn.evaluate(X_val, y_val, verbose=0)[1]        # validation accuracy
print(round(cnn_acc, 4))                                  # "the CNN is 1.5 points better than the fully connected network"

**Run, and you should see:** Version A prints two numbers, the first one high; Version B stops with a red error that ends with `output.shape=(None, 5, 5, 10)` (or `(32, 5, 5, 10)`); Version C, if you run it, prints a validation accuracy higher than in Part 2 (about 0.987).

**Write down:** one line per version: what it printed, then *wrong* or *right*, and a few words why.

*Your answer:* 

**Think about** (with your neighbour, no writing): Versions A and C give numbers that look fine. If you had not read the code line by line, how could you have noticed?

> **Going further (optional)**
>
> - Write your own version D that differs from Part 2 in one line, runs without an error and is wrong (for example, something to do with `x_test`, `move` or `X_val`). Give it to your neighbour. Can they find the line from the code alone? From the score alone?


## After class: how today's lab helps with Assignment 2

| What you did today | Where it appears in Assignment 2 |
|---|---|
| Part 1: images as 28 × 28 × 1, `Conv2D`, the number of parameters | Extension task A: "input 28 × 28 × 1, at least two `Conv2D` layers"; Table A: the number of parameters |
| Part 2: the same epochs and seed, the seconds taken by `fit`, the move of 2 pixels | Extension task A: Table A, and "shift the images 2 pixels to the right" |
| Part 4: the 5 most common mistakes, 10 misclassified images | Section 4, **Table 5** and **Figure 3** |
| Part 5: the three mistakes | none of them may appear in your assignment |

Three things are different in the assignment.

1. In Section 4, Table 5 and Figure 3 use **`test`**, once, with the predictions of your final model. Today we used the validation set, which is fine for learning.
2. Extension task A asks for the same number of epochs and the same seed as Section 4: 20 epochs, seed 0. Today we used 2 epochs to save time.
3. The assignment's data block uses `fetch_openml`; the Keras file gives the same images in the same order.

## Reflection

One or two sentences each.

1. What does a convolution layer learn when it is trained, and where does Keras keep it?
2. Version C of Part 5 gave a better score than Part 2. Why is it still wrong?
3. When you compare two networks, what four things would you report, and what must be the same for both?

# Appendix: only if MNIST did not download

Use this only if the last line of the Part 0 set-up cell stopped with a download error. Each cell below gives exactly the same 60,000 images and digits; run one, then go on with the data block (from `X = x_train.reshape(...)`).

**Backup 1: the course's copy of the same file**, on the course website.

In [ ]:
BACKUP_URL = "https://ml.wxing.me/assignments/assignment-two/mnist.npz"
request = urllib.request.Request(BACKUP_URL, headers={"User-Agent": "Mozilla/5.0"})   # the website only answers browsers
download = urllib.request.urlopen(request)                # ask for the file (about 11 MB)
open("mnist.npz", "wb").write(download.read())            # save it: the same file Keras uses, from a different server
full = np.load("mnist.npz")                               # four arrays: x_train, y_train, x_test, y_test
x_train = full["x_train"]                                 # 60,000 images of 28 x 28
y_train = full["y_train"]                                 # their digits
print(x_train.shape)                                      # images, height, width

**Backup 2: the official file, downloaded directly** (the one Keras uses, without Keras's own download step).

In [ ]:
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
urllib.request.urlretrieve(MNIST_URL, "mnist.npz")        # all 70,000 MNIST images, about 11 MB
full = np.load("mnist.npz")                               # four arrays: x_train, y_train, x_test, y_test
x_train = full["x_train"]                                 # 60,000 images of 28 x 28
y_train = full["y_train"]                                 # their digits
print(x_train.shape)                                      # images, height, width

**Run, and you should see:** `(60000, 28, 28)` from either cell.